In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/System-Threat-Forecaster/sample_submission.csv
/kaggle/input/System-Threat-Forecaster/train.csv
/kaggle/input/System-Threat-Forecaster/test.csv


DUMMY SUBMISSION

In [2]:
df=pd.read_csv("/kaggle/input/System-Threat-Forecaster/train.csv")
X_test=pd.read_csv("/kaggle/input/System-Threat-Forecaster/test.csv")
df.shape
df.head(5)
X=df.drop("target", axis=1)
y=df["target"]
from sklearn.dummy import DummyClassifier
model = DummyClassifier().fit(X,y)
y_pred=model.predict(X_test)
submission = pd.DataFrame({"id":range(0,X_test.shape[0]), "target": y_pred})
submission.to_csv("submission.csv", index=False)

TRIAL 1 (Random Forest Model)

In [3]:
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score
from sklearn.preprocessing import LabelEncoder
df = pd.read_csv("/kaggle/input/System-Threat-Forecaster/train.csv")
X_test = pd.read_csv("/kaggle/input/System-Threat-Forecaster/test.csv")
print(f"Training data shape: {df.shape}")
print(f"Test data shape: {X_test.shape}")
X = df.drop(["target", "MachineID"], axis=1, errors='ignore')  
y = df["target"]
X_test = X_test.drop("MachineID", axis=1, errors='ignore') 
categorical_features = X.select_dtypes(include=['object']).columns
encoder_dict = {}
for col in categorical_features:
    encoder = LabelEncoder()
    X[col] = encoder.fit_transform(X[col].astype(str))
    encoder_dict[col] = encoder  
    X_test[col] = X_test[col].apply(lambda x: encoder.transform([x])[0] if x in encoder.classes_ else -1)
X.fillna(-1, inplace=True)
X_test.fillna(-1, inplace=True)
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)
model = RandomForestClassifier(
    n_estimators=100,
    max_depth=10,
    random_state=42,
    n_jobs=-1
)

Training data shape: (100000, 76)
Test data shape: (10000, 75)


Milestone 1

In [4]:
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score
from sklearn.preprocessing import LabelEncoder
df = pd.read_csv("/kaggle/input/System-Threat-Forecaster/train.csv")
X_test = pd.read_csv("/kaggle/input/System-Threat-Forecaster/test.csv")
unique_os_versions = pd.concat([df['OSVersion'], X_test['OSVersion']]).nunique()
print(f"Number of unique OS versions: {unique_os_versions}")


Number of unique OS versions: 7


In [5]:
max_value = df["NumAntivirusProductsInstalled"].max()
print(f"Maximum value of NumAntivirusProductsInstalled: {max_value}")

Maximum value of NumAntivirusProductsInstalled: 5.0


In [6]:
gamer_malware_count = df[(df["IsGamer"] == 1) & (df["target"] == 1)].shape[0]
print(f"Number of systems owned by gamers where malware was detected: {gamer_malware_count}")

Number of systems owned by gamers where malware was detected: 16294


In [7]:
filtered_data = df[df["IsPassiveModeEnabled"] == 1]
most_frequent_value = filtered_data["RealTimeProtectionState"].mode()[0]
print(f"Most frequent value of RealTimeProtectionState: {most_frequent_value}")

Most frequent value of RealTimeProtectionState: 0.0


In [8]:
resolution_count = df[
    (df["PrimaryDisplayResolutionHorizontal"] == 1366) & 
    (df["PrimaryDisplayResolutionVertical"] == 768)
].shape[0]

print(f"Number of systems with a screen resolution of 1366 x 768: {resolution_count}")

Number of systems with a screen resolution of 1366 x 768: 51435


In [9]:
median_ram = df["TotalPhysicalRAMMB"].median()
print(f"50th percentile (median) value of TotalPhysicalRAMMB: {median_ram}")

50th percentile (median) value of TotalPhysicalRAMMB: 4096.0


TRIAL 2

In [10]:
import pandas as pd
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.metrics import roc_auc_score
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import StratifiedKFold

# Load the data
df = pd.read_csv("/kaggle/input/System-Threat-Forecaster/train.csv")
X_test = pd.read_csv("/kaggle/input/System-Threat-Forecaster/test.csv")

# Print data shapes
print(f"Training data shape: {df.shape}")
print(f"Test data shape: {X_test.shape}")

# Prepare the data
X = df.drop(["target", "MachineID"], axis=1, errors='ignore')  
y = df["target"]
X_test = X_test.drop("MachineID", axis=1, errors='ignore') 

# Encode categorical features
categorical_features = X.select_dtypes(include=['object']).columns
encoder_dict = {}
for col in categorical_features:
    encoder = LabelEncoder()
    X[col] = encoder.fit_transform(X[col].astype(str))
    encoder_dict[col] = encoder  
    X_test[col] = X_test[col].apply(lambda x: encoder.transform([x])[0] if x in encoder.classes_ else -1)

# Handle missing values
X.fillna(-1, inplace=True)
X_test.fillna(-1, inplace=True)

# Train-test split
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)

# Model configuration
model = GradientBoostingClassifier(
    n_estimators=100,
    max_depth=3,
    random_state=42
)

# Cross-validation for better validation score
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
cv_auc_scores = cross_val_score(model, X, y, cv=cv, scoring='roc_auc')

# Print cross-validation AUC scores and mean
print(f"Cross-validation AUC scores: {cv_auc_scores}")
print(f"Mean AUC score from cross-validation: {cv_auc_scores.mean():.4f}")

# Train the model on the full dataset
model.fit(X_train, y_train)

# Validate the model on the validation set
y_val_pred = model.predict_proba(X_val)[:, 1]
val_auc = roc_auc_score(y_val, y_val_pred)
print(f"Validation AUC: {val_auc:.4f}")

# Generate test predictions
y_test_pred = model.predict(X_test)

# Save submission
submission = pd.DataFrame({"id": range(0, X_test.shape[0]), "target": y_test_pred})
submission.to_csv("submission.csv", index=False)
print("Submission file saved as 'submission.csv'")


Training data shape: (100000, 76)
Test data shape: (10000, 75)
Cross-validation AUC scores: [0.67136205 0.6758702  0.66455795 0.66434085 0.66276856]
Mean AUC score from cross-validation: 0.6678
Validation AUC: 0.6704
Submission file saved as 'submission.csv'
